<a href="https://colab.research.google.com/github/MariuszMleczak/Projekt_AI_-_Klasyfikator_defektow_powierzchni_stalowych/blob/main/prezentacja_wytrenowanego_modelu_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
import gradio as gr
from PIL import Image

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
weights = ResNet50_Weights.DEFAULT
model = models.resnet50(weights=None)

model.fc = nn.Linear(model.fc.in_features, 6)

model.load_state_dict(torch.load(
    "/content/drive/MyDrive/AI_PROJECT/Wytrenowany_model_AI/neu_model.pth",
    map_location=device
))

model = model.to(device)
model.eval()

print("Model loaded")

Model loaded


In [ ]:
classes = [
    "crazing",
    "inclusion",
    "patches",
    "pitted_surface",
    "rolled-in_scale",
    "scratches"
]

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [ ]:
def predict(image):
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)

    return f"{classes[pred.item()]} ({conf.item()*100:.2f}%)"

In [ ]:
interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Klasyfikator defektów powierzchni stalowych",
    description="Wgraj zdjęcie uszkodzonej powierzchni stalowej i naciśnij 'submit'."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b23be505b48692d4a5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
